In [86]:
from data_loader import load_raw_data
orders, order_items, products, customers, categories_translation = load_raw_data()
import pandas as pd

In [87]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')

In [88]:
date_cols = ['order_approved_at', 'order_purchase_timestamp', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

In [89]:
orders[date_cols] = orders[date_cols].apply(pd.to_datetime)

In [90]:
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

In [91]:
orders.groupby('order_status')['order_id'].count()

order_status
approved           2
canceled         625
created            5
delivered      96478
invoiced         314
processing       301
shipped         1107
unavailable      609
Name: order_id, dtype: int64

In [92]:
categories_translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [93]:
order_items.shape

(112650, 7)

In [94]:
orders['order_id'].is_unique


True

In [95]:
products['product_id'].is_unique

True

In [96]:
categories_translation['product_category_name'].is_unique

True

In [97]:
customers['customer_id'].is_unique

True

In [98]:
df= order_items.merge(orders, on= 'order_id', how= 'left')


In [99]:
df.shape

(112650, 14)

In [100]:
df= df.merge(products, on= 'product_id', how= 'left')

In [101]:
df.shape

(112650, 22)

In [102]:
df= df.merge(categories_translation, on= 'product_category_name', how= 'left')

In [103]:
df.shape

(112650, 23)

In [104]:
df= df.merge(customers, on= 'customer_id', how= 'left')

In [105]:
df.shape

(112650, 27)

In [106]:
df.isna().sum()

order_id                            0
order_item_id                       0
product_id                          0
seller_id                           0
shipping_limit_date                 0
price                               0
freight_value                       0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  15
order_delivered_carrier_date     1194
order_delivered_customer_date    2454
order_estimated_delivery_date       0
product_category_name            1603
product_name_lenght              1603
product_description_lenght       1603
product_photos_qty               1603
product_weight_g                   18
product_length_cm                  18
product_height_cm                  18
product_width_cm                   18
product_category_name_english    1627
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_sta

In [107]:
order_items.shape[0]

112650

In [108]:
df.shape[0]

112650

In [109]:
df['product_category_name_english'].isna().sum()

np.int64(1627)

In [110]:
missing_eng = df['product_category_name_english'].isna()
no_eng = df[missing_eng]
no_eng['product_category_name'].value_counts(dropna=False)

product_category_name
NaN                                              1603
portateis_cozinha_e_preparadores_de_alimentos      15
pc_gamer                                            9
Name: count, dtype: int64

In [111]:
df['product_category_name_english'] = df['product_category_name_english'].fillna(df['product_category_name'])

In [112]:
df['product_category_name_english'] = df['product_category_name_english'].replace('portateis_cozinha_e_preparadores_de_alimentos', 'portable_kitchen_appliances')

In [113]:
df['product_category_name_english'] = df['product_category_name_english'].fillna('unknown')

In [114]:
df['product_category_name_english'].isna().sum()

np.int64(0)

In [115]:
df['product_category_name_english'].value_counts()

product_category_name_english
bed_bath_table               11115
health_beauty                 9670
sports_leisure                8641
furniture_decor               8334
computers_accessories         7827
                             ...  
cds_dvds_musicals               14
la_cuisine                      14
pc_gamer                         9
fashion_childrens_clothes        8
security_and_services            2
Name: count, Length: 74, dtype: int64

1627 items (1.4%) had no Engish category: 1603 had no category at all, so i labelled them uknown; 24 had untranslated names, so i kept or translated them

In [116]:
df['revenue'] = df['price']

In [117]:
df['revenue'].head()

0     58.90
1    239.90
2    199.00
3     12.99
4    199.90
Name: revenue, dtype: float64

In [118]:
df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M')

In [119]:
df['order_month'].head()

0    2017-09
1    2017-04
2    2018-01
3    2018-08
4    2017-02
Name: order_month, dtype: period[M]

In [120]:
df['delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

In [121]:
df['delivery_days'].head()

0     7.0
1    16.0
2     7.0
3     6.0
4    25.0
Name: delivery_days, dtype: float64

In [122]:
delivered = df['order_delivered_customer_date'].notna()

In [123]:
df['is_late'] = df['order_delivered_customer_date'] > df['order_estimated_delivery_date']

In [124]:
df['is_late'].value_counts()

is_late
False    103935
True       8715
Name: count, dtype: int64

In [125]:
df['delivery_days'].describe()

count    110196.000000
mean         12.007723
std           9.451455
min           0.000000
25%           6.000000
50%          10.000000
75%          15.000000
max         209.000000
Name: delivery_days, dtype: float64

In [126]:
df[delivered]['is_late'].mean()

np.float64(0.07908635522160513)

In [127]:
df = df.rename(columns={'product_category_name_english':'category', 'order_purchase_timestamp':'purchase_at'})

In [128]:
df

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,purchase_at,...,product_width_cm,category,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,revenue,order_month,delivery_days,is_late
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,...,14.0,cool_stuff,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,58.90,2017-09,7.0,False
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,...,40.0,pet_shop,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP,239.90,2017-04,16.0,False
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,...,33.0,furniture_decor,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG,199.00,2018-01,7.0,False
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,...,15.0,perfumery,af861d436cfc08b2c2ddefd0ba074622,12952,atibaia,SP,12.99,2018-08,6.0,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,...,30.0,garden_tools,64b576fb70d441e8f1b2d7d446e483c5,13226,varzea paulista,SP,199.90,2017-02,25.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41,b51593916b4b8e0d6f66f2ae24f2673d,delivered,2018-04-23 13:57:06,...,40.0,housewares,0c9aeda10a71f369396d0c04dce13a64,65077,sao luis,MA,299.99,2018-04,17.0,False
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53,84c5d4fbaf120aae381fad077416eaa0,delivered,2018-07-14 10:26:46,...,38.0,computers_accessories,0da9fe112eae0c74d3ba1fe16de0988b,81690,curitiba,PR,350.00,2018-07,9.0,False
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95,29309aa813182aaddc9b259e31b870e6,delivered,2017-10-23 17:07:56,...,19.0,sports_leisure,cd79b407828f02fdbba457111c38e4c4,4039,sao paulo,SP,99.90,2017-10,4.0,False
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72,b5e6afd5a41800fdf401e0272ca74655,delivered,2017-08-14 23:02:59,...,20.0,computers_accessories,eb803377c9315b564bdedad672039306,13289,vinhedo,SP,55.99,2017-08,1.0,False


In [129]:
keep_cols  = ['order_id', 'customer_unique_id','order_status','purchase_at', 'order_month','category', 'revenue','freight_value', 'customer_city', 'customer_state', 'delivery_days', 'is_late']
sales = df[keep_cols]

In [130]:
sales.to_csv('data/processed/sales.csv', index=False)

In [131]:
df1 = orders.merge(customers, on= 'customer_id', how= 'left')

In [132]:
df1 = df1.rename(columns={'order_purchase_timestamp':'purchase_at'})

In [133]:
df1.columns

Index(['order_id', 'customer_id', 'order_status', 'purchase_at',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state'],
      dtype='str')

In [134]:
keep_cols1 = ['order_id','order_status', 'purchase_at','customer_state']
orders_status = df1[keep_cols1]

In [135]:
orders_status.to_csv('data/processed/orders_status.csv', index=False)